# 00 - GPU smoke test

Confirms that the single GKE node's L4 is visible to the container, that PyTorch
can use it, and roughly what throughput it delivers. Run every cell top to bottom.

`02-deploy-jupyter.sh` copies this notebook into `work/`, which is backed by a
PersistentVolumeClaim, so edits survive a pod restart.

## 1. Driver and device, straight from the node

In [1]:
!nvidia-smi

Sun Sep  6 18:26:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.173.02             Driver Version: 580.173.02     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   48C    P8             18W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Topology matrix. On a single-GPU node this is trivial, but the same command is
# how you confirm PCIe vs NVLink paths once there is more than one GPU per node.
!nvidia-smi topo -m

	GPU0	CPU Affinity	NUMA Affinity	GPU NUMA ID
GPU0	 X 	0-7	0		N/A

Legend:

  X    = Self
  SYS  = Connection traversing PCIe as well as the SMP interconnect between NUMA nodes (e.g., QPI/UPI)
  NODE = Connection traversing PCIe as well as the interconnect between PCIe Host Bridges within a NUMA node
  PHB  = Connection traversing PCIe as well as a PCIe Host Bridge (typically the CPU)
  PXB  = Connection traversing multiple PCIe bridges (without traversing the PCIe Host Bridge)
  PIX  = Connection traversing at most a single PCIe bridge
  NV#  = Connection traversing a bonded set of # NVLinks


## 2. PyTorch sees the GPU

In [3]:
import torch

print('torch          :', torch.__version__)
print('cuda available :', torch.cuda.is_available())
print('cuda runtime   :', torch.version.cuda)
print('device count   :', torch.cuda.device_count())

assert torch.cuda.is_available(), 'No CUDA device. Check the pod has nvidia.com/gpu: 1.'

props = torch.cuda.get_device_properties(0)
print()
print('name           :', props.name)
print('capability     : sm_%d%d' % (props.major, props.minor))
print('total memory   : %.1f GiB' % (props.total_memory / 1024**3))
print('SM count       :', props.multi_processor_count)
print('bf16 supported :', torch.cuda.is_bf16_supported())

torch          : 2.11.0+cu128
cuda available : True
cuda runtime   : 12.8
device count   : 1

name           : NVIDIA L4
capability     : sm_89
total memory   : 22.0 GiB
SM count       : 58
bf16 supported : True


## 3. Matmul throughput

A crude but honest check that the GPU is actually doing work. An L4 is rated at
roughly 120 TFLOP/s dense bf16; a real measurement in the 60-100 range is normal
for this size, and anything near 1 means you are silently running on CPU.

In [4]:
import time
import torch

n, iters = 8192, 50
a = torch.randn(n, n, device='cuda', dtype=torch.bfloat16)
b = torch.randn(n, n, device='cuda', dtype=torch.bfloat16)

for _ in range(10):          # warm up: first calls include kernel autotuning
    a @ b
torch.cuda.synchronize()

start = time.perf_counter()
for _ in range(iters):
    a @ b
torch.cuda.synchronize()
elapsed = time.perf_counter() - start

flops = 2 * n**3 * iters     # one multiply-add per output element per k
print('%.1f ms/matmul' % (elapsed / iters * 1e3))
print('%.1f TFLOP/s bf16' % (flops / elapsed / 1e12))

19.6 ms/matmul
56.2 TFLOP/s bf16


## 4. NCCL initialises

Single process, single rank, so this measures nothing about the network. It only
proves the NCCL library loads and the process group forms, which is the thing that
breaks first when you later scale to multiple nodes.

In [5]:
import os
import torch
import torch.distributed as dist

os.environ.setdefault('MASTER_ADDR', '127.0.0.1')
os.environ.setdefault('MASTER_PORT', '29500')
os.environ.setdefault('RANK', '0')
os.environ.setdefault('WORLD_SIZE', '1')

if not dist.is_initialized():
    dist.init_process_group(backend='nccl')

torch.cuda.set_device(0)
t = torch.ones(1024, device='cuda')
dist.all_reduce(t)
torch.cuda.synchronize()

print('world size :', dist.get_world_size())
print('all_reduce :', t[0].item(), '(expected 1.0 at world size 1)')

dist.destroy_process_group()

all_reduce : 1.0 (expected 1.0 at world size 1)


## 5. Where the pod is running

Useful once there is more than one node and you need to know which one you landed on.

In [6]:
import socket
import subprocess

print('pod hostname :', socket.gethostname())
print()
print(subprocess.run(['bash', '-lc', 'cat /proc/cpuinfo | grep -c ^processor'],
                     capture_output=True, text=True).stdout.strip(), 'vCPU visible')
print(subprocess.run(['bash', '-lc', "free -g | awk '/Mem:/ {print $2}'"],
                     capture_output=True, text=True).stdout.strip(), 'GiB RAM visible')

pod hostname : jupyter-5f6f79887f-z2fx4

8 vCPU visible
31 GiB RAM visible
